In [30]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os, json, pickle
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from go_ml.models.warmup_esm_finetune import ESMFinetune
from go_ml.data_utils import *
from argparse import ArgumentParser
import transformers

import pickle
data_path = "/home/andrew/GO_interp/data/train_esm_datasets/"
with open(f"{data_path}/train_dataset.pkl", "rb") as f:
    train_dataset = pickle.load(f)
with open(f"{data_path}/val_dataset.pkl", "rb") as f:
    val_dataset = pickle.load(f)

from go_ml.data_utils import prot_func_collate
checkpoint_path = "/home/andrew/GO_interp/checkpoints/esm_finetune_fin.ckpt"
model = ESMFinetune.load_from_checkpoint(checkpoint_path, strict=False)

In [33]:
from go_ml.data_utils import prot_func_collate, ProtFuncDataset

from go_ml.eval_utils import filter_annot_df
data_root = '../gen_datasets/datasets'
csa_df = filter_annot_df(pd.read_csv(f'{data_root}/csa_annot.csv', sep='\t'))
llps_df = filter_annot_df(pd.read_csv(f'{data_root}/llps_dataset.csv', sep='\t'))
elms_df = filter_annot_df(pd.read_csv(f'{data_root}/elms_dataset.csv', sep='\t'))
ds_labels = ['csa', 'llps', 'elms']

import json
train_path = "/home/andrew/cafa5_team/data/"
with open(f"{train_path}/cafa_dataset/go_terms.json", "r") as f:
    go_terms = json.load(f)

In [36]:
csa_ds = ProtFuncDataset.from_annot_df(csa_df, go_terms)
llps_ds = ProtFuncDataset.from_annot_df(llps_df, go_terms)
elms_ds = ProtFuncDataset.from_annot_df(elms_df, go_terms)

csa_dl = DataLoader(csa_ds, batch_size=12, collate_fn=prot_func_collate, shuffle=False)
llps_dl = DataLoader(llps_ds, batch_size=12, collate_fn=prot_func_collate, shuffle=False)
elms_dl = DataLoader(elms_ds, batch_size=12, collate_fn=prot_func_collate, shuffle=False)

In [54]:
from sklearn.metrics import f1_score
preds_dict = {}
labels_dict = {}
for ds, dl, ds_label in zip([csa_ds, llps_ds, elms_ds], [csa_dl, llps_dl, elms_dl], ds_labels):
    print(f"Evaluating {ds_label} dataset")
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in dl:
            batch = {k: v.to(model.device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            seq_ind, seq_mask = batch['seq_ind'], batch['mask']
            preds = model(seq_ind, seq_mask)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(batch['labels'].cpu().numpy())
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    preds_dict[ds_label] = torch.tensor(all_preds)
    labels_dict[ds_label] = torch.tensor(all_labels)
    f1 = f1_score(all_labels, all_preds > 0, average='micro')
    print(f"F1 Score for {ds_label} dataset: {f1:.4f}")

Evaluating csa dataset
F1 Score for csa dataset: 0.0324
Evaluating llps dataset
F1 Score for llps dataset: 0.0136
Evaluating elms dataset
F1 Score for elms dataset: 0.0622
